# Threads 貼文資料處理

這個 Notebook 負責讀取 `範例.txt`，並將其非結構化的文字內容，轉換為結構化的 DataFrame 格式。

In [5]:
import re
import pandas as pd

def parse_threads_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # 步驟一：用分隔線將主要貼文切開
    posts = content.split('------------------------------')
    
    structured_data = []
    
    for post in posts:
        post = post.strip()
        if not post:
            continue
            
        # 步驟二：萃取貼文編號
        post_num_match = re.search(r'【貼文 (\d+)】', post)
        if not post_num_match:
            continue
        post_num = post_num_match.group(1)
        
        # 步驟三：過濾 Meta Data (網址、內容：、作者、日期)
        # 尋找內文開始的地方：通常在日期 (如 2025-5-14) 之後的換行
        content_start_match = re.search(r'內容：\n.*?\n\d{4}-\d{1,2}-\d{1,2}\n', post)
        if content_start_match:
            post_text = post[content_start_match.end():]
        else:
            # 備用方案：如果沒匹配到日期，就先抓整個內容
            post_text = post
            
        # 步驟四：使用正規表示式切割對話串 (Threads)
        # 匹配類似:
        # （續 
        # 1
        # /
        # 3
        # 或是結尾的
        # 3
        # /
        # 3
        marker_pattern = r'(?:（續\s*\n)?\s*\d+\s*\n\s*/\s*\n\s*\d+\s*'
        
        # 將貼文依據標記切割成多串對話
        threads = re.split(marker_pattern, post_text)
        
        # 過濾掉因為切割產生的空白字串，並去除前後多餘空白
        threads = [t.strip() for t in threads if t.strip()]
        
        # 步驟五：組裝資料
        for i, thread_content in enumerate(threads):
            thread_id = f"貼文{post_num}_第{i+1}串"
            structured_data.append({
                "貼文與串文編號": thread_id,
                "文字內容": thread_content
            })
            
    return pd.DataFrame(structured_data)

# 執行函式並顯示結果
df = parse_threads_file('WhiteTalk.txt')
df

,貼文與串文編號,文字內容
0,貼文1_第1串,（吐槽文，不喜勿入）\n之前看到某個財經暢銷書的教授\n出來點出技術分析的爭議\n我也陸續發...
1,貼文1_第2串,人性就是這麼懶\n尤其是那種規則越固定、越不用動腦的系統，越棒\n也最能PUA那些金融市場的...
2,貼文1_第3串,抱歉，我尊重市場，不尊重你的無腦套路\n所以啊，別再用技術圖表來坑騙新手了\n市場不是你那幾...
3,貼文2_第1串,最近有時間就來整理一些以前被問到的問題\n我發現剛接觸option的新人\n喜歡套 Blac...
4,貼文2_第2串,這種風險，不是平均值能描述的\n就像波動性也不是常態分布能描述的\n而且這類風險你很難觀察得...
...,...,...
1228,貼文392_第4串,外匯市場更分散，也沒有像股票那樣的撮合機制，每個人拿到的資訊也不對等\n有些人是央行、有些是...
1229,貼文392_第5串,比較有趣的發現是，如果你依照這個邏輯去分類，把前一天報酬率排序（高、中、低），再把異常量排序...
1230,貼文392_第6串,畢竟Spot 是市場上最直接的價格反映，出事的人最容易在這邊出手\nForward 是有邏輯...
1231,貼文392_第7串,當然啦，在這裡我必須補充一句\n我並不是在鼓勵大家去照抄這套策略，也不是說你看到Sharpe...


In [6]:
# 檢視資料是否符合預期，也可以匯出成 CSV 檔
df.to_csv('structured_threads.csv', index=False, encoding='utf-8-sig')
df.head()

,貼文與串文編號,文字內容
0,貼文1_第1串,（吐槽文，不喜勿入）\n之前看到某個財經暢銷書的教授\n出來點出技術分析的爭議\n我也陸續發...
1,貼文1_第2串,人性就是這麼懶\n尤其是那種規則越固定、越不用動腦的系統，越棒\n也最能PUA那些金融市場的...
2,貼文1_第3串,抱歉，我尊重市場，不尊重你的無腦套路\n所以啊，別再用技術圖表來坑騙新手了\n市場不是你那幾...
3,貼文2_第1串,最近有時間就來整理一些以前被問到的問題\n我發現剛接觸option的新人\n喜歡套 Blac...
4,貼文2_第2串,這種風險，不是平均值能描述的\n就像波動性也不是常態分布能描述的\n而且這類風險你很難觀察得...
